In [306]:
import pandas as pd
import numpy as np
import os

import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.svm import SVC
from statsmodels.tsa.tsatools import lagmat

# Definición de colores oficiales aproximados de Wenia
WENIA_PURPLE = '#1C003D'   # Violeta base muy oscuro
WENIA_MAGENTA = '#9033FF'  # Morado vibrante
WENIA_LIME = "#67DAF7"     # Verde Lima Neón (Acento)
WENIA_GRAY = '#F4F4F9'     # Fondo sutil

wenia_template = go.layout.Template()

wenia_template.layout = {
    'paper_bgcolor': 'white',
    'plot_bgcolor': 'white',
    'font': {'color': WENIA_PURPLE, 'family': 'Aptos, sans-serif', 'size': 12},
    'xaxis': {
        'showgrid': False, 
        'linecolor': WENIA_PURPLE,
        'tickfont': {'color': WENIA_PURPLE},
        'dtick': "M4", 
    },
    'yaxis': {
        'showgrid': True, 
        'gridcolor': '#EAEAEA',
        'linecolor': WENIA_PURPLE,
        'zeroline': True,
        'zerolinecolor': WENIA_PURPLE
    },
    'legend': {
        'orientation': 'h',    
        'yanchor': 'bottom',   
        'y': 1.02,             
        'xanchor': 'right',
        'x': 1
    },
    'colorway': [WENIA_PURPLE, WENIA_LIME, WENIA_MAGENTA]
}

pio.templates['wenia_theme'] = wenia_template
pio.templates.default = 'wenia_theme'

ROOTH_PATH = os.getcwd()
FILTE_PATH = os.path.join(ROOTH_PATH, 'data.xlsx')


In [85]:
data = pd.read_excel(
    FILTE_PATH,
    index_col = 'Date',
    parse_dates = ['Date'],
    dtype = {
        'Open':float,
        'High':float,
        'Low':float,
        'Close':float,
        'Adj Close':float, 
        'Volume':int
    }
).sort_index()
data.describe()

,Open,High,Low,Close,Adj Close,Volume
count,3.444000e+03,3.444000e+03,3.444000e+03,3.444000e+03,3.444000e+03,3.444000e+03
mean,1.077325e+06,1.054616e+06,1.128315e+06,1.059863e+06,1.046638e+06,1.612499e+10
std,2.517456e+06,2.516051e+06,2.572977e+06,2.507732e+06,2.490546e+06,1.895212e+10
min,2.000500e+02,0.000000e+00,1.715100e+02,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.142961e+04,1.080075e+04,1.128749e+04,1.057586e+04,1.058134e+04,1.160000e+08
50%,3.834397e+04,3.740610e+04,3.762339e+04,3.658042e+04,3.659119e+04,1.070000e+10
75%,3.896608e+05,3.799960e+05,3.920695e+05,3.802650e+05,3.780990e+05,2.630000e+10
max,9.936561e+06,9.996743e+06,9.992007e+06,9.951519e+06,9.934434e+06,3.510000e+11


# Handling 0.0 values

While we could apply various data cleaning techniques such as interpolation, averaging, or the last observation carried forward, I have decided to omit them. This is due to simplicity and the fact that these methods can alter data integrity and reliability, potentially leading to erroneous hypotheses—especially in time-series data where, under certain conditions and over short periods, there are no significant variations. Furthermore, addressing this would require a separate, comprehensive study that tends to be exhaustive.

In [111]:
df = data[data.columns[data.columns != 'Volume']].replace(0.0, None).dropna().astype(float)
df.insert(0,'TypicalPrice',(df['High']+df['Low']+df['Close'])/3)

# Handling Outliers
Ordenando los datos por el índice y graficando podemos notar que hay un período hasta antes de septiembre de 2020 con datos erráticos, algo extraño en series de tiempo financieras, dado que no tengo acceso a información externa que pueda brindar un mayor contexto (take over, splits,...) es difícil proponer una normalización de los datos que no afecte la integridad de los mismos, generándo distribuciónes y por ende hiótesis erróneas, de esta manera opté por preservar solo los datos apartir de abril de 2017 en adelane

In [112]:
df = df.loc['2020-10-01':]

# Time Series Analysis

In [ ]:
def RSI(price:pd.Series, period:int=14) -> pd.Series:
    delta = price.diff()

    gain = (delta.where(delta > 0, 0))
    loss = (-delta.where(delta < 0, 0))


    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))
    
    return rsi

In [307]:
typical_price = df['TypicalPrice']
df_returns = np.log(typical_price/typical_price.shift(1)).dropna()
df_volatility = pow(df_returns - df_returns.mean(),2)
df_rsi = RSI(typical_price)

fig = make_subplots(rows=4,cols=1, shared_xaxes=True)
fig.add_trace(
    go.Scatter(x=df.index, y = df['TypicalPrice'], mode='lines',name='Typical Price'), row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df_rsi.index, y=df_rsi, mode='lines', name='RSI'), row=2, col=1
)

fig.add_trace(
    go.Scatter(x=df_returns.index,y=df_returns, mode='lines',name='Log return'), row=3, col=1
)

fig.add_trace(
    go.Scatter(x=df_volatility.index, y=df_volatility, mode='lines', name='Volatility'), row=4, col=1
)


# Líneas de referencia para RSI (70 y 30)
fig.add_hline(y=70, line_dash="dash", line_color="#CBCBCC", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color='#CBCBCC', row=2, col=1)


fig.update_layout(
    height=450,      
    width=550,
    xaxis = {
        'dtick': "M6",        
    },
           
    xaxis2 = {
        'dtick': "M6", 
    }, 
           
    xaxis3 = {
        'dtick': "M6", 
    }, 
           
    xaxis4 = {
        'dtick': "M6", 
    },        
    title={
        'text': 'Time Series Identification',
        'x': 0.1,      # Superior Izquierda
        'y': 0.95,
        'xanchor': 'left'
    },
    margin=dict(l=50, r=20, t=75, b=25), # Márgenes optimizados
    showlegend=True # En formato columna, la leyenda suele estorbar
)

# Aplicar color de fuente de los títulos de subplots según tu estilo
for i in fig['layout']['annotations']:
    i['font'] = dict(size=12, color=WENIA_PURPLE)

fig.show()


 # Bollinger Bands Strategy

A useful strategy in sideways trends is Bollinger Bands; the strategy consists of buying when the price breaks the lower band and selling at the moving average. It is done analogously with the upper band by taking a short position.




In [300]:
def BollingerBands(price: pd.Series, lags: int = 20, deviations: float = 2.0) -> pd.DataFrame:
    sma = price.rolling(window=lags).mean()
    std = price.rolling(window=lags).std()
    
    df = pd.DataFrame({
        'Price': price,
        'SMA': sma,
        'UpperBound': sma + (deviations * std),
        'LowerBound': sma - (deviations * std)
    }).dropna()
    
    signals = []
    current_pos = 0 

    for i in range(len(df)):
        signal = 0
        price_t = df['Price'].iloc[i]
        upper_t = df['UpperBound'].iloc[i]
        lower_t = df['LowerBound'].iloc[i]

        if price_t >= upper_t:
            if current_pos == 0 or current_pos == 1:
                signal = -1
                current_pos += -1
        
        if price_t <= lower_t:
            if current_pos == 0 or current_pos == -1:
                signal = 1
                current_pos += 1 

        signals.append(signal)

    df['Position'] = signals
    df['logRet'] = np.log(df['Price'] / df['Price'].shift(1))
    df['Strategy_LogRet'] = df['Position'].shift(1) * df['logRet']
    df['Cumulative_Ret'] = df['Strategy_LogRet'].cumsum().apply(np.exp)

    return df


typical_price = df['TypicalPrice']
bb_strategy = BollingerBands(price=typical_price, lags=20, deviations=2.0)

buy_signals = bb_strategy[bb_strategy['Position']==1]['Price']
sell_signals = bb_strategy[bb_strategy['Position']==-1]['Price']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True)

fig.add_trace(
    go.Scatter(x = bb_strategy.index, y = bb_strategy['Price'], mode='lines', name=None, showlegend=False), row=1, col=1
)

fig.add_trace(
    go.Scatter(x = bb_strategy.index, y = bb_strategy['UpperBound'], mode='lines', name=None,line=dict(color='salmon'),showlegend=False), row=1, col=1
)

fig.add_trace(
    go.Scatter(x = bb_strategy.index, y = bb_strategy['LowerBound'], mode='lines', name=None,line=dict(color="#3F7AFA"),showlegend=False), row=1, col=1
)

fig.add_trace(
    go.Scatter(x = buy_signals.index, y = buy_signals, mode='markers', name=None,
               marker=dict(symbol='triangle-up', size=12, color='green', line=dict(width=1, color='black')),showlegend=False), row=1, col=1
)

fig.add_trace(
    go.Scatter(x = sell_signals.index, y = sell_signals, mode='markers', name='',
               marker=dict(symbol='triangle-down', size=12, color='red', line=dict(width=1, color='black')),showlegend=False), row=1, col=1
)


fig.add_trace(
    go.Scatter(x = bb_strategy.index, y = bb_strategy['Cumulative_Ret'], mode='lines', name='Cummulative Return', line=dict(color='#9033FF' )), row=2, col=1
)

fig.update_layout(
        title={
        'text': 'Bollinger Bands Trading Strategy',
        'x': 0.05,          # 0 es izquierda, 1 es derecha
        'y': 0.9,       # Ajusta la altura para que no toque el borde
        'xanchor': 'left',
        'yanchor': 'top'
    }
)


fig.update_layout(
    height=450,      
    width=550,
    xaxis = {
        'dtick': "M6", 
    },     
    xaxis2 = {
        'dtick': "M6", 
    },       
    title={
        'text': 'Bollinger Bands Trading Strategy',
        'x': 0.1,      
        'y': 0.95,
        'xanchor': 'left'
    },
    margin=dict(l=50, r=20, t=75, b=25), 
    showlegend=True 
)


for i in fig['layout']['annotations']:
    i['font'] = dict(size=12, color=WENIA_PURPLE)

fig.show()



# Support Vector Machine (SVM) Strategy

SVM allows to classify under a hyperplane two signals: 1 prices goes up -1 prices goes down

In [243]:

def add_lags(serie:pd.DataFrame,lags):
    df = pd.DataFrame(serie)
    for lag in range(1,lags+1):
        df['lag_{}'.format(lag)] = serie.shift(lag)
    df.dropna(inplace=True)
    return(df)

def SVM(price:pd.Series, lags:int=5):
    returns = np.log(price/price.shift(1))

    df = pd.DataFrame({
        'Price' : price,
        'LogRet' : returns
    })

    df_lags = add_lags(df['LogRet'],lags=lags)

    y = np.sign(df_lags['LogRet'])
    X = np.sign(df_lags[df_lags.columns[df_lags.columns != 'LogRet']])

    svm = SVC(C=30).fit(X,y) 

    prediction = svm.predict(X)

    signals = []
    current_pos = 0

    for i in prediction:
        signal = 0
        if current_pos == 0:
            signal = i
            current_pos += i
        else:
            if current_pos != i:
                signal = i
                current_pos += i 
        

        signals.append(signal)

    df = df.iloc[lags+1:,:]
    df['Position'] = signals
    df['Strategy_LogRet'] = df['Position'].shift(1) * df['LogRet']
    df['Cumulative_Ret'] = df['Strategy_LogRet'].cumsum().apply(np.exp)
    return df

In [301]:
typical_price = df['TypicalPrice']
svm_strategy = SVM(typical_price, lags=8)


buy_signals = svm_strategy[svm_strategy['Position']==1]['Price']
sell_signals = svm_strategy[svm_strategy['Position']==-1]['Price']

fig = make_subplots(rows=2, cols=1,shared_xaxes=True)

fig.add_trace(
    go.Scatter(x = svm_strategy.index, y = svm_strategy['Price'], mode='lines', name='TypicalPrice',showlegend=False), row=1, col=1
)


fig.add_trace(
    go.Scatter(x = buy_signals.index, y = buy_signals, mode='markers', name='Buy', showlegend=False,
               marker=dict(symbol='triangle-up', size=12, color='green', line=dict(width=1, color='black'),)), row=1, col=1
)

fig.add_trace(
    go.Scatter(x = sell_signals.index, y = sell_signals, mode='markers', name='Sell', showlegend=False,
               marker=dict(symbol='triangle-down', size=12, color='red', line=dict(width=1, color='black'))), row=1, col=1
)


fig.add_trace(
    go.Scatter(x = svm_strategy.index, y = svm_strategy['Cumulative_Ret'], mode='lines', name='Cummulative Return', line=dict(color='#9033FF')), row=2, col=1
)

fig.update_layout(
        title={
        'text': 'SVM Trading Strategy',
        'x': 0.05,          # 0 es izquierda, 1 es derecha
        'y': 0.9,       # Ajusta la altura para que no toque el borde
        'xanchor': 'left',
        'yanchor': 'top'
    }
)

fig.update_layout(
    height=450,      
    width=550,
    xaxis = {
        'dtick': "M6", 
    },     
    xaxis2 = {
        'dtick': "M6", 
    },       
    title={
        'text': 'SVM Trading Strategy',
        'x': 0.1,      
        'y': 0.95,
        'xanchor': 'left'
    },
    margin=dict(l=50, r=20, t=75, b=25), 
    showlegend=True 
)


for i in fig['layout']['annotations']:
    i['font'] = dict(size=12, color=WENIA_PURPLE)
fig.show()


# Backtesting metrics

In [251]:
## total return
bb_total_ret, svm_total_ret = bb_strategy['Cumulative_Ret'].iloc[-1], svm_strategy['Cumulative_Ret'].iloc[-1]
## Sharpe ratio
bb_sharpe = (bb_strategy['Strategy_LogRet'].mean()/bb_strategy['Strategy_LogRet'].std()) * np.sqrt(252)
svm_sharpe = (svm_strategy['Strategy_LogRet'].mean()/svm_strategy['Strategy_LogRet'].std()) * np.sqrt(252)
## Drawdown máximo
bb_max_dd = (bb_strategy['Cumulative_Ret']/bb_strategy['Cumulative_Ret'].cummax()-1).min()
svm_max_dd = (svm_strategy['Cumulative_Ret']/svm_strategy['Cumulative_Ret'].cummax()-1).min()

data_metrics = {
    'Bollinger (GBM+GARCH)': [
        bb_total_ret - 1,  # Retorno porcentual
        bb_sharpe, 
        bb_max_dd
    ],
    'SVM (Lags Matrix)': [
        svm_total_ret - 1, 
        svm_sharpe, 
        svm_max_dd
    ]
}

df_comparativo = pd.DataFrame(
    data_metrics, 
    index=['Total Return', 'Sharpe Ratio', 'Max Drawdown']
)

# df_reporte = df_comparativo.style.format({
#     'Bollinger (GBM+GARCH)': "{:.2%}" if 'Ratio' not in df_comparativo.index else "{:.2f}",
#     'SVM (Lags Matrix)': "{:.2%}"
# })

# print(df_comparativo)